# RunPod vLLM generation with saved results
Use one RTX 4090 24 GB, On-Demand, an official RunPod PyTorch template with **Python 3.11 or 3.12**, JupyterLab and persistent `/workspace` storage (about 50 GB). Upload this notebook only. It downloads the user's existing ZIP directly from Drive, verifies it, converts the actual frozen Q4 base and trained LoRA, and uses vLLM for generation.

This is a separate FP16 numerical replication with eight concurrent prediction state machines. No retraining is performed. Mac/MLX runs remain separate. Validate smoke and the 45 held-out development outputs, then review throughput before starting a full paid benchmark. GPU execution on RunPod has not been tested during local preparation.

Results are saved per output, checkpointed every five minutes, and exported to `/workspace/nlp-runpod-vllm-results.zip`. This is persistent-volume storage, not an automatic Google Drive upload. Download backups before terminating the Pod. Closing the notebook does not stop GPU billing.

In [ ]:
from pathlib import Path, PurePosixPath
import sys, subprocess, json, zipfile, hashlib, time, os, signal
assert sys.platform.startswith('linux'), 'Use a Linux RunPod GPU Pod'
assert (3, 11) <= sys.version_info[:2] < (3, 13), 'Choose a RunPod template with Python 3.11 or 3.12'
assert Path('/workspace').is_dir(), 'Mount persistent storage at /workspace'
subprocess.run(['nvidia-smi'], check=True)
DRIVE_URL = 'https://drive.google.com/file/d/1e0G7nG1B8c6G4CKzw6-U69my-x7GJfuD/view?usp=sharing'
BUNDLE = Path('/workspace/nlp-colab-assets.zip')
EXPECTED_BUNDLE_SHA256 = 'fbe767cf2ed584134ce822204ae46e70461b7000cc01b51bcccb84f2eba3a926'
ROOT = Path('/workspace/nlp-runpod-vllm')
VENV = Path('/workspace/nlp-vllm-env')
STATE = Path('/workspace/nlp-vllm-state')
RESULTS_ZIP = Path('/workspace/nlp-runpod-vllm-results.zip')
LAUNCHER_REPO = Path('/workspace/nlp-vllm-launcher')
REPO_URL = 'https://github.com/faisalAI27/Qwen2.5-1.5B-finetune.git'
NEW_FILES = ('scripts/runpod_vllm.py', 'src/s1small/vllm_backend.py', 'src/s1small/vllm_export.py',
             'configs/runpod_vllm_v1.json', 'configs/vllm_prompt_reference_v1.json', 'requirements-vllm.txt')
LAUNCHER_COMMIT = '1cae805a2a29b7ba769838380f2fd71f058fa824'

## 1. Install the pinned vLLM environment
The small launcher repository and legacy asset source use independent pinned commits. This environment does not use the Colab/MLX requirements or their CUDA header workarounds. Download/install time is also billed while the Pod is running.

In [ ]:
if not (LAUNCHER_REPO / '.git').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(LAUNCHER_REPO)], check=True)
subprocess.run(['git', 'checkout', '--detach', LAUNCHER_COMMIT], cwd=LAUNCHER_REPO, check=True)
subprocess.run(['git', 'diff', '--quiet', 'HEAD', '--', *NEW_FILES], cwd=LAUNCHER_REPO, check=True)
environment = subprocess.run([sys.executable, '-m', 'venv', str(VENV)], capture_output=True, text=True)
if environment.returncode:
    print(environment.stdout, environment.stderr)
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'virtualenv==20.36.1'], check=True)
    subprocess.run([sys.executable, '-m', 'virtualenv', '--no-periodic-update', str(VENV)], check=True)
PYTHON = str(VENV / 'bin/python')
subprocess.run([PYTHON, '-m', 'pip', 'install', '--no-cache-dir', '-r', str(LAUNCHER_REPO / 'requirements-vllm.txt')], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'check'], check=True)

## 2. Download the user's Drive ZIP and verify both code versions
The Drive link must permit anonymous download. If Drive quota or permissions prevent it, upload the same ZIP to `/workspace/nlp-colab-assets.zip` and rerun this cell. The exact SHA-256 is checked before extraction. No benchmark answers are included in this transfer.

In [ ]:
if not BUNDLE.is_file():
    download = (
        'import gdown; result = gdown.download('
        f'url={DRIVE_URL!r}, output={str(BUNDLE)!r}, '
        'fuzzy=True, resume=True, use_cookies=False); '
        'assert result, "Drive download failed; upload the original ZIP manually"'
    )
    subprocess.run([PYTHON, '-c', download], check=True)
hasher = hashlib.sha256()
with BUNDLE.open('rb') as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b''):
        hasher.update(chunk)
assert hasher.hexdigest() == EXPECTED_BUNDLE_SHA256, 'ZIP differs from the verified original; do not extract it'
with zipfile.ZipFile(BUNDLE) as archive:
    bundled = archive.read('colab_bundle_manifest.json')
    metadata = json.loads(bundled)
    if metadata['bundle_mode'] == 'assets_with_github_code':
        assert metadata['source_commit'], 'Asset source commit is required'
        if not (ROOT / '.git').exists():
            subprocess.run(['git', 'clone', REPO_URL, str(ROOT)], check=True)
        subprocess.run(['git', 'checkout', '--detach', metadata['source_commit']], cwd=ROOT, check=True)
    ROOT.mkdir(parents=True, exist_ok=True)
    old = ROOT / 'colab_bundle_manifest.json'
    assert not old.exists() or old.read_bytes() == bundled, 'Use fresh workspace/state directories for a different asset bundle'
    for member in archive.infolist():
        relative = PurePosixPath(member.filename)
        assert not relative.is_absolute() and '..' not in relative.parts, 'Unsafe archive path'
    archive.extractall(ROOT)
original = ROOT / 'manifests/evaluation_freeze_macos.json'
canonical = ROOT / 'manifests/evaluation_freeze.json'
if canonical.exists() and canonical.read_bytes() == original.read_bytes():
    canonical.rename(ROOT / 'manifests/evaluation_freeze_macos_from_clone.json')
launcher_hashes = {}
for name in NEW_FILES:
    content = (LAUNCHER_REPO / name).read_bytes()
    target = ROOT / name
    target.parent.mkdir(parents=True, exist_ok=True)
    assert not target.exists() or target.read_bytes() == content, f'Changed launcher file {name}; use a fresh workspace for new code'
    target.write_bytes(content)
    launcher_hashes[name] = hashlib.sha256(content).hexdigest()
launcher_manifest = {'launcher_commit': LAUNCHER_COMMIT, 'asset_source_commit': metadata['source_commit'],
                     'asset_zip_sha256': EXPECTED_BUNDLE_SHA256, 'files_sha256': launcher_hashes}
launcher_path = ROOT / 'vllm_launcher_manifest.json'
assert not launcher_path.exists() or json.loads(launcher_path.read_text()) == launcher_manifest, 'Launcher identity differs'
launcher_path.write_text(json.dumps(launcher_manifest, indent=2) + '\n')
(ROOT / 'VLLM_ISOLATED_WORKSPACE').write_text('Independent RunPod vLLM numerical replication.\n')
STATE.mkdir(parents=True, exist_ok=True)
identity = STATE / 'IDENTITY.json'
if identity.exists():
    assert json.loads(identity.read_text()) == launcher_manifest, 'Checkpoint folder belongs to different code/assets'
else:
    assert not (STATE / 'LATEST.json').exists(), 'Unidentified checkpoint folder; select a fresh STATE folder'
    identity.write_text(json.dumps(launcher_manifest, indent=2) + '\n')
for name, relative in {'HF_HOME':'cache/huggingface', 'VLLM_CACHE_ROOT':'cache/vllm',
                       'TORCHINDUCTOR_CACHE_DIR':'cache/torchinductor', 'TRITON_CACHE_DIR':'cache/triton',
                       'XDG_CACHE_HOME':'cache/xdg'}.items():
    os.environ[name] = str(ROOT / relative)
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_USE_V1'] = '1'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
(ROOT / 'results').mkdir(exist_ok=True)
(ROOT / 'results/installed-packages.txt').write_text(subprocess.check_output([PYTHON, '-m', 'pip', 'freeze'], text=True))
subprocess.run([PYTHON, str(ROOT / 'scripts/colab_checkpoint.py'), 'restore', '--drive-folder', str(STATE)], check=True)
subprocess.run([PYTHON, str(ROOT / 'scripts/runpod_vllm.py'), 'verify'], check=True)
print('Original asset hashes and all 15 frozen prompt/token references verified')

## 3. Save, export, monitor and run stages
Completed answers are saved immediately by the runner in atomic slots, durable attempts and an fsynced JSONL log. This helper additionally snapshots mutable state and refreshes a downloadable results ZIP every five minutes and after each stage. It prints new log lines and actual GPU activity while generation runs.

A same-volume snapshot is not an external backup. Download the results ZIP and a state snapshot before terminating the Pod. The ZIP excludes multi-GB weights/environments but includes conversion provenance, code and all available results. Grading/analysis should run on the Pod before final download.

In [ ]:
def export_results():
    temporary = RESULTS_ZIP.with_suffix('.zip.incomplete')
    with zipfile.ZipFile(temporary, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as archive:
        for folder in ('results', 'manifests'):
            for path in (ROOT / folder).rglob('*'):
                if path.is_file() and not path.name.endswith(('.lock', '.incomplete')):
                    archive.write(path, str(path.relative_to(ROOT)))
        for name in (*NEW_FILES, 'vllm_launcher_manifest.json', 'colab_bundle_manifest.json',
                     'artifacts/vllm-export/export_manifest.json'):
            path = ROOT / name
            if path.is_file():
                archive.write(path, name)
    temporary.replace(RESULTS_ZIP)

def save_state():
    subprocess.run([PYTHON, str(ROOT / 'scripts/colab_checkpoint.py'), 'save', '--drive-folder', str(STATE)], check=True)
    export_results()
    print('Results saved:', RESULTS_ZIP, flush=True)

def run_stage(mode):
    log = ROOT / 'results' / ('vllm-' + mode + '.log')
    position = log.stat().st_size if log.exists() else 0
    def show_new():
        nonlocal position
        with log.open('rb') as reader:
            reader.seek(position)
            content = reader.read()
            position = reader.tell()
        if content:
            print(content.decode(errors='replace')[-10000:], end='', flush=True)
    def gpu_sample():
        sample = subprocess.run(['nvidia-smi', '--query-gpu=timestamp,utilization.gpu,utilization.memory,memory.used,power.draw', '--format=csv,noheader'], capture_output=True, text=True)
        if sample.returncode == 0:
            print('GPU activity:', sample.stdout.strip(), flush=True)
            with (ROOT / 'results/vllm-gpu.csv').open('a') as stream:
                stream.write(sample.stdout)
    with log.open('ab', buffering=0) as stream:
        process = subprocess.Popen([PYTHON, str(ROOT / 'scripts/runpod_vllm.py'), mode], cwd=ROOT,
            stdin=subprocess.DEVNULL, stdout=stream, stderr=subprocess.STDOUT, start_new_session=True)
        checkpoint_due = time.monotonic() + 300
        try:
            while process.poll() is None:
                time.sleep(30)
                show_new()
                if process.poll() is None:
                    if mode in ('smoke', 'development', 'test'):
                        gpu_sample()
                    else:
                        print(mode + ' is still running (CPU/setup stage)', flush=True)
                if time.monotonic() >= checkpoint_due:
                    save_state()
                    checkpoint_due = time.monotonic() + 300
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=20)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
                process.wait()
            save_state()
            raise
    show_new()
    save_state()
    if process.returncode:
        print('\n'.join(log.read_text(errors='replace').splitlines()[-100:]))
        raise RuntimeError(f'{mode} exited {process.returncode}; saved evidence: {log}')
    print(mode + ': completed successfully', flush=True)

## 4. Convert the trained model (CPU; once)
Reconstruct the frozen Q4 base as dense FP16 shards and export the exact FP32 LoRA tensors in PEFT layout. No original full-precision model is downloaded, and no optimizer/training runs. Conversion checks the original trained-adapter fingerprint. A completed conversion is verified/reused on rerun; interrupted tensor exports resume.

In [ ]:
run_stage('convert')

## 5. vLLM GPU smoke
Checks counted stop tokens and a real held-out answer before allowing evaluation. A successful smoke verifies unchanged exported files; it does not establish mathematical accuracy or Mac token parity.

In [ ]:
run_stage('smoke')

## 6. Development: 45 outputs with concurrent generation
Five held-out questions × nine conditions, up to eight concurrent prediction state machines feeding vLLM. Wait for a complete summary and inspect **aggregate_tokens_per_second_this_invocation**. Individual latency includes concurrent scheduling and is not isolated single-request latency. Checkpoint/ZIP saving happens automatically while the stage runs.

In [ ]:
run_stage('development')

## 7. Freeze the validated vLLM configuration
Requires 45 outputs, matching prompt/protocol/model/runtime identities, observed readout and the unchanged grader. Reports exact-token parity with Mac for transparency; FP16/kernel/batching differences are expected to be possible. This creates an independent vLLM freeze.

In [ ]:
run_stage('freeze')
run_stage('status')

## 8. Full benchmark (paid GPU time)
Run after reviewing actual development throughput and the current hourly charge. This plans 4,500 independent outputs under **math500-runpod-vllm-v1**. Saved completed predictions from this exact runtime are retained on resume. Do not copy Mac, Colab or MLX predictions into this run.

In [ ]:
run_stage('test')

## 9. Grade, analyze and download
Run grading/analysis before final backup. Results include raw responses, tokens, extraction/grading outcomes, timings, runtime/provenance and plots. Partial runs retain explicit denominators. Download `/workspace/nlp-runpod-vllm-results.zip` and the latest state snapshot to your Mac or Drive, then stop the Pod in the console. The notebook does not stop GPU/storage billing.

In [ ]:
run_stage('grade')
run_stage('analyze')
run_stage('status')
save_state()
print('Download results:', RESULTS_ZIP)
print('State snapshots:', sorted(STATE.glob('cuda-state-*.zip')))
print('Stop the Pod in the RunPod console after backing up results. Storage remains billed while retained.')